In [ ]:
import numpy as np
import plotly.graph_objects as go
import os

# ---- Config ----
scan_index = 1  # change to load planck_1.npy, planck_2.npy, etc.

npy_dir = r"planck_scans/npy"
scaled_path = r"experiments/geotransformer.faces.stage4.gse.k3.max.oacl.stage2.sinkhorn/plank_scaled.npy"

# ---- Load ----
src_points = np.load(os.path.join(npy_dir, f"planck_{scan_index}.npy")).astype(np.float64)
scaled_points = np.load(scaled_path)

# ---- Plot ----
all_points = np.vstack([src_points, scaled_points])
center = (all_points.min(0) + all_points.max(0)) / 2
extent = (all_points.max(0) - all_points.min(0)).max() / 2

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=src_points[:, 0], y=src_points[:, 1], z=src_points[:, 2],
    mode='markers', marker=dict(size=2, color='steelblue'),
    name=f"planck_{scan_index} (raw)"
))

fig.add_trace(go.Scatter3d(
    x=scaled_points[:, 0], y=scaled_points[:, 1], z=scaled_points[:, 2],
    mode='markers', marker=dict(size=2, color='tomato'),
    name="plank_scaled"
))

fig.update_layout(
    scene=dict(
        xaxis=dict(range=[center[0]-extent, center[0]+extent]),
        yaxis=dict(range=[center[1]-extent, center[1]+extent]),
        zaxis=dict(range=[center[2]-extent, center[2]+extent]),
        aspectmode='cube'
    )
)

fig.show()

In [ ]:
import numpy as np
import os

npy_dir = r"planck_scans/npy"
scaled_path = r"planck_scans/plank_scaled.npy"

scaled = np.load(scaled_path)
scaled_span = scaled.max(0) - scaled.min(0)
scaled_max_span = scaled_span.max()

files = sorted([f for f in os.listdir(npy_dir) if f.endswith('.npy')])

print(f"{'Name':<22} {'N pts':>8}  {'span_x':>8} {'span_y':>8} {'span_z':>8}  {'scale_factor':>12}")
print("-" * 80)
for fname in files:
    pts = np.load(os.path.join(npy_dir, fname))
    span = pts.max(0) - pts.min(0)
    factor = scaled_max_span / span.max()
    print(f"{fname:<22} {len(pts):>8}  {span[0]:>8.3f} {span[1]:>8.3f} {span[2]:>8.3f}  {factor:>12.5f}")

print("-" * 80)
print(f"{'plank_scaled (ref)':<22} {len(scaled):>8}  {scaled_span[0]:>8.3f} {scaled_span[1]:>8.3f} {scaled_span[2]:>8.3f}  {'1.00000':>12}")
print()
print("scale_factor = multiply planck_scans/npy points by this to match plank_scaled max span")


In [ ]:
GLOBAL_FACTOR = 0.01

In [ ]:
import numpy as np
import os

npy_dir = r"planck_scans/npy"
out_dir = r"planck_scans/npy_scaled"
scaled_path = r"experiments/geotransformer.faces.stage4.gse.k3.max.oacl.stage2.sinkhorn/plank_scaled.npy"
target_n = 2000
rng = np.random.default_rng(42)

os.makedirs(out_dir, exist_ok=True)

scaled_ref = np.load(scaled_path)
ref_max_span = (scaled_ref.max(0) - scaled_ref.min(0)).max()

files = sorted([f for f in os.listdir(npy_dir) if f.endswith('.npy')])

for f_num, fname in enumerate(files):
    pts = np.load(os.path.join(npy_dir, fname)).astype(np.float64)
   #factor = ref_max_span / (pts.max(0) - pts.min(0)).max()
    factor = GLOBAL_FACTOR
    pts = pts * factor
    pts = pts - pts.mean(0)
    idx = rng.choice(len(pts), size=target_n, replace=False)
    pts = pts[idx]
    out_path = os.path.join(out_dir, f"planck_{f_num+1}.npy")
    np.save(out_path, pts.astype(np.float32))
    print(f"{fname}: scale={factor:.5f}  ->  {pts.shape}  centroid={pts.mean(0).round(4)}")

print(f"\nSaved {len(files)} files to {out_dir}")


In [ ]:
# ---- Config ----
scan_index = 10  # change to load planck_1.npy, planck_2.npy, etc.

npy_dir = r"planck_scans/npy_scaled"
scaled_path = r"planck_scans/plank_scaled.npy"

# ---- Load ----
src_points = np.load(os.path.join(npy_dir, f"planck_{scan_index}.npy")).astype(np.float64)
scaled_points = np.load(scaled_path)

# ---- Plot ----
all_points = np.vstack([src_points, scaled_points])
center = (all_points.min(0) + all_points.max(0)) / 2
extent = (all_points.max(0) - all_points.min(0)).max() / 2

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=src_points[:, 0], y=src_points[:, 1], z=src_points[:, 2],
    mode='markers', marker=dict(size=2, color='steelblue'),
    name=f"planck_{scan_index} (raw)"
))

fig.add_trace(go.Scatter3d(
    x=scaled_points[:, 0], y=scaled_points[:, 1], z=scaled_points[:, 2],
    mode='markers', marker=dict(size=2, color='tomato'),
    name="plank_scaled"
))

fig.update_layout(
    scene=dict(
        xaxis=dict(range=[center[0]-extent, center[0]+extent]),
        yaxis=dict(range=[center[1]-extent, center[1]+extent]),
        zaxis=dict(range=[center[2]-extent, center[2]+extent]),
        aspectmode='cube'
    )
)

fig.show()